# 16 z-loss 解决什么问题，如何实现？

## 面试回答主线

z-loss 在交叉熵之外惩罚 $z=\log\sum\exp(\text{logits})$ 的绝对大小，常用于抑制 softmax partition 的无界漂移、改善大规模低精度训练稳定性。它不是让所有 logits 变小，更不是替代准确率目标；系数过大可能压低置信度、伤害主任务。面试回答要区分 logit shift 不变的交叉熵与 z-loss 对绝对 partition 的约束。实验用六条意图分类 logits 比较纯 CE 与 CE+z-loss 的梯度，并构造过大系数导致主损失被压制的失败。

**核心公式：** $L=L_{CE}+\lambda_z(\log\sum_j\exp z_j)^2$。交叉熵对同一行 logits 加常数近似不变，但 z-loss 会惩罚这种整体上移。

下面按真实案例、基线、手写机制、结果表和失败修复组织回答；所有数据都是可复现的教学实验。


## 真实案例

场景是客服与账户安全系统中的六条脱敏离线事件。字段包含工单文本、有效 token 数和风险标签；它们模拟真实的数据结构，但样本极小，只用于观察公式和状态变化。


In [1]:
import math  # 导入数学函数以实现训练与掩码公式。
import warnings  # 导入警告控制模块保持输出干净。
warnings.filterwarnings('ignore', message='The pynvml package is deprecated')  # 屏蔽环境依赖的非教学弃用提示。
import torch  # 导入张量计算和自动微分能力。
import torch.nn as nn  # 导入模块基类以手写网络结构。
torch.manual_seed(29)  # 固定随机种子使教学输出可复现。
torch.set_num_threads(1)  # 限制小实验 CPU 线程数。
samples = [  # 构造六条脱敏客服对话作为真实语义样本。
    {'id': 'C01', 'text': '支付重复扣款，申请退款', 'tokens': 6, 'risk': 1},  # 资金风险工单。
    {'id': 'C02', 'text': '收不到登录验证码', 'tokens': 2, 'risk': 0},  # 登录支持工单。
    {'id': 'C03', 'text': '账户有陌生转账记录', 'tokens': 5, 'risk': 1},  # 账户安全工单。
    {'id': 'C04', 'text': '修改订单收货地址', 'tokens': 3, 'risk': 0},  # 售后咨询工单。
    {'id': 'C05', 'text': '银行卡盗刷需要冻结', 'tokens': 7, 'risk': 1},  # 高优先级安全工单。
    {'id': 'C06', 'text': '更正发票抬头信息', 'tokens': 4, 'risk': 0},  # 账单服务工单。
]  # 结束教学数据定义。
features = torch.tensor([[1.0, 0.0, 1.0], [0.0, 1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 1.0], [1.0, 1.0, 0.0], [0.0, 1.0, 1.0]])  # 构造三维可解释特征。
labels = torch.tensor([1, 0, 1, 0, 1, 0])  # 构造风险分类标签。
print('教学实验：六条脱敏离线客服事件，只验证机制，不代表线上收益。')  # 声明数据边界。
for row in samples:  # 逐条展示真实语义输入。
    print(f"{row['id']} | token={row['tokens']} | risk={row['risk']} | {row['text']}")  # 输出样本字段。
print(f'特征形状={tuple(features.shape)}，标签={labels.tolist()}')  # 输出张量形状。


教学实验：六条脱敏离线客服事件，只验证机制，不代表线上收益。
C01 | token=6 | risk=1 | 支付重复扣款，申请退款
C02 | token=2 | risk=0 | 收不到登录验证码
C03 | token=5 | risk=1 | 账户有陌生转账记录
C04 | token=3 | risk=0 | 修改订单收货地址
C05 | token=7 | risk=1 | 银行卡盗刷需要冻结
C06 | token=4 | risk=0 | 更正发票抬头信息
特征形状=(6, 3)，标签=[1, 0, 1, 0, 1, 0]


## Baseline / 基线

先在同一批六条事件上运行最简单方案。基线不是稻草人，它提供固定的输入、口径和可比较指标。


In [2]:
logits = torch.tensor([[8.0, 6.0], [5.0, 9.0], [10.0, 7.0], [4.0, 8.0], [9.0, 5.0], [3.0, 7.0]], requires_grad=True)  # 构造六条工单的高绝对值 logits。
ce_loss = torch.nn.functional.cross_entropy(logits, labels)  # 计算只关注相对类别差的交叉熵基线。
baseline_metric = float(ce_loss)  # 保存主任务 loss。
print(f'纯 CE={baseline_metric:.4f}，平均 logsumexp={float(torch.logsumexp(logits, dim=-1).mean()):.3f}')  # 展示绝对 partition 很大但 CE 很小的现象。


纯 CE=3.5414，平均 logsumexp=8.541


## 手写核心实现与中间量

代码保留关键分子分母、mask、梯度、参数组或重算路径，而不让 Trainer 或高层框架隐藏面试问题本身。


In [3]:
log_partition = torch.logsumexp(logits, dim=-1)  # 手写读取每条样本的 logsumexp。
z_penalty = log_partition.pow(2).mean()  # 计算 z-loss 惩罚项。
total_loss = ce_loss + 1e-3 * z_penalty  # 用小系数将 z-loss 加到主目标。
gradient = torch.autograd.grad(total_loss, logits)[0]  # 获取 CE+z-loss 对 logits 的真实梯度。
core_metric = float(z_penalty)  # 保存 z-loss 数值供结果表比较。
print(f'z-loss={core_metric:.4f}，总 loss={float(total_loss):.4f}，首条 logsumexp={float(log_partition[0]):.3f}')  # 输出核心分项和中间量。
print(f'首条 logits 梯度={ [round(float(value), 4) for value in gradient[0]] }')  # 展示 z-loss 对绝对尺度的额外梯度。


z-loss=73.8700，总 loss=3.6152，首条 logsumexp=8.127
首条 logits 梯度=[0.1492, -0.1465]


In [4]:
comparison_rows = [('Baseline', float(baseline_metric)), ('核心机制', float(core_metric))]  # 建立同一口径的结果表。
for name, metric in comparison_rows:  # 逐行输出结果。
    print(f'{name:<8} | 指标={metric:.6f}')  # 展示可读数值对照。


Baseline | 指标=3.541353
核心机制     | 指标=73.870049


## 结果解读

基线和核心输出只在本受控案例中比较。生产中应记录 CE、z-loss、logsumexp 分布和溢出计数，并按模型规模、精度和数据配方调小系数；不能只报总 loss。 观察结果时应关注中间量是否符合公式，而不是把六条样本上的数字宣传为线上收益。

## 失败案例

下一个单元故意破坏关键假设，并用实现修复证明该假设为何必要。


In [5]:
excessive_total = ce_loss + 1.0 * z_penalty  # 故意把 z-loss 系数放大一千倍。
failure_metric = float(excessive_total / ce_loss)  # 计算总目标被正则项主导的倍数。
fixed_total = ce_loss + 1e-3 * z_penalty  # 恢复小的教学系数。
fix_metric = float(fixed_total / ce_loss)  # 计算修复后总目标相对主任务的比例。
print(f'失败：lambda=1 时总/CE={failure_metric:.1f}；修复：lambda=1e-3 时总/CE={fix_metric:.3f}')  # 展示系数不能脱离主任务量级。


失败：lambda=1 时总/CE=21.9；修复：lambda=1e-3 时总/CE=1.021


## 工程取舍、常见坑与延伸追问

**工程取舍：** 生产中应记录 CE、z-loss、logsumexp 分布和溢出计数，并按模型规模、精度和数据配方调小系数；不能只报总 loss。

**常见坑：** 把 logsumexp 当成 logits 均值，或给过大的 z-loss 系数导致模型为了减 z 而牺牲分类/生成质量。

**延伸追问：** z-loss 与 logit clipping、RMSNorm、BF16 overflow 有何关系？为何需要分别记录总 loss 和各分项？

## 生产差距

本 Notebook 在 CPU/FP32 下处理 6 条离线事件，省略了真实 token packing、分布式同步、混合精度、checkpoint、隐私治理、监控告警和灰度回滚。生产版本必须替换为受审计的数据管道与系统级指标。


In [6]:
assert core_metric > 0.0  # 验证高绝对值 logits 产生非零 z-loss。
assert failure_metric > fix_metric  # 验证过大 z-loss 系数会主导目标。
assert torch.isfinite(gradient).all()  # 验证总目标梯度数值有限。
assert baseline_metric > 0.0  # 验证交叉熵主任务有效。
